#Notebook used to clean recollected data
###The csv need to be uploaded into the content folder


Packages instalation

In [2]:
pip install -qq openai python-dotenv

####Imports needed to clean the texts

In [3]:
import pandas as pd
import os
from dotenv import load_dotenv
import openai
import time
from google.colab import files
import textwrap

###Load of the CSV and showing it for a better comparison.
###You can load a previous run csv to start from that index so you don´t have to start from zero.

In [4]:
output_path = "cleaned_result.csv"
df = pd.read_csv("textoselegidos.csv") #Put your csv with the texts you want to clean
df

,Species,Text,Page_id,Volume,Year,Source,Source_ID,Institution,Language,Rights,Copyright
0,Transandinomys talamancae,"Identification. Small, head-body length 107 mm...",64171116,72,2023,Internet Archive,bonnzoologicalbv72izoola,Smithsonian Libraries and Archives,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
1,Transandinomys talamancae,known that Oecomys bicolor represents a comple...,64171116,72,2023,Internet Archive,bonnzoologicalbv72izoola,Smithsonian Libraries and Archives,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
2,Transandinomys talamancae,Figure 3. Phylogenetic tree of maximum likelih...,64738101,6,2022,Virtual Item,vi210912v6no1202220250518010740,Pensoft Publishers,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
3,Transandinomys talamancae,Identification. Dorsum fur grayish with reddis...,64171121,72,2023,Internet Archive,bonnzoologicalbv72izoola,Smithsonian Libraries and Archives,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
4,Transandinomys talamancae,Identification. Dorsal fur short (8-10 mm) and...,64171121,72,2023,Internet Archive,bonnzoologicalbv72izoola,Smithsonian Libraries and Archives,English,https://biodiversitylibrary.org/permissions,In copyright. Digitized with the permission of...
...,...,...,...,...,...,...,...,...,...,...,...
148,Jacana spinosa,It is recorded that among Jacanas the role of ...,48072849,61.0,1964.0,Internet Archive,journalofbombay611964bomb,Smithsonian Libraries and Archives,English,https://www.biodiversitylibrary.org/permissions/,In Copyright. Digitized with the permission of...
149,Jacana spinosa,The coast of western North America is the home...,39732414,NaN,NaN,Internet Archive,distributionmigr00cook,Library of Congress,English,NaN,Not provided. Contact Holding Institution to v...
150,Jacana spinosa,The Mexican jacana was described originally fr...,39732414,NaN,NaN,Internet Archive,distributionmigr00cook,Library of Congress,English,NaN,Not provided. Contact Holding Institution to v...
151,Jacana spinosa,"The spotted sandpiper, Actitis macularia (Fig....",18775477,NaN,NaN,Internet Archive,collegezoology00hegnuoft,University of Toronto - Gerstein Science Infor...,English,NaN,NOT_IN_COPYRIGHT


###Checking if the API KEY is well charged

In [5]:
load_dotenv()
client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("API Key ready to use")

API Key ready to use


###The definition of all the functions used to cleaning the texts in the dataframe
####As it´s name says, the clean_text function is the one who uses the API KEY and asks the Api to clean the text
####The next one is the one who access the data in the dataframe and build the new csv, every 25 requests it downloads the current csv to the user's device.



In [13]:
def clean_text(text, language='english'):
    prompt = f"The language is {language} and the text is:\n\n{text}"

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": "You are a helpful assistant that improves the clarity and correctness of text."},
            {"role": "user",  "content": "Clean and correct the following text. Try to maintain coherence and remove everything that refers to a figure or image, such as (Fig 11), only focus in the ones between parenthesis I will first send you the language—either Spanish or English—and then the text. I only need the cleaned version of the text."},
            {"role": "assistant", "content": "Perfect. Go ahead and send me the language first, and then the text. I'll return only the cleaned and corrected version."},
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )
    return response.choices[0].message.content.strip()

def clean_dataframe(df, output_path):
    start_index = 0
    if os.path.exists(output_path):
        try:
            existing = pd.read_csv(output_path)
            start_index = len(existing[existing['Cleaned_Text'].notna()])
            df['Cleaned_Text'] = existing.get('Cleaned_Text', pd.NA)
        except:
            df['Cleaned_Text'] = pd.NA
    else:
        df['Cleaned_Text'] = pd.NA

    # Ask user how many rows to process
    total_rows = len(df) - start_index
    print(f"\nThere are {total_rows} rows left to clean starting from index {start_index}.")
    while True:
        try:
            max_rows = int(input("How many rows would you like to process? (Enter a positive integer): "))
            if max_rows <= 0:
                print("Please enter a number greater than 0.")
            else:
                break
        except ValueError:
            print("Invalid input. Please enter an integer.")

    end_index = min(start_index + max_rows, len(df))

    for i in range(start_index, end_index):
        text = str(df.at[i, 'Text'])
        language = str(df.at[i, 'Language']).lower()
        if pd.isna(text) or text.strip() == "":
            continue
        print(f"Cleaning row {i} ({language})...")
        try:
            cleaned = clean_text(text, language)
        except Exception as e:
            print(f"Error: {e}")
            cleaned = None
        df.at[i, 'Cleaned_Text'] = cleaned
        time.sleep(1.2)
        if (i + 1) % 10 == 0:
            df.to_csv(output_path, index=False)
            print(f"Downloading backup at row {i}")
            files.download(output_path)

    df.to_csv(output_path, index=False)
    print("Cleaning completed.")

def review_cleaned_rows(df):
    for i in range(len(df)):
        original = str(df['Text'].iloc[i])
        cleaned = str(df['Cleaned_Text'].iloc[i])

        if pd.isna(cleaned) or original.strip() == "":
            continue  # Skip if text is empty or not yet cleaned

        print(f"\n--- Row {i} ---")
        print("\nOriginal Text:\n")
        print(textwrap.fill(original, width=100))

        print("\nCleaned Text:\n")
        print(textwrap.fill(cleaned, width=100))

        while True:
            cont = input("\nDo you want to continue to the next row? (y/n): ").strip().lower()
            if cont in ['y', 'n']:
                break
            print("Please enter 'y' or 'n'.")

        if cont == 'n':
            print("Stopping review.")
            break

###The call of the fuction that cleans the dataframe, since the function changes or creates the csv we only have to make it a dataframe to verify the new cleaned texts.

In [18]:
clean_dataframe(df, output_path)
df = pd.read_csv("cleaned_result.csv")
#review_cleaned_rows(df) #this is to make a review of how good the cleaning was.


There are 143 rows left to clean starting from index 10.


KeyboardInterrupt: Interrupted by user

##I leave the review function in a cell in case you forgot to uncomment it in the last cell.
###To make things clear, if you say no it will finish the review and download the csv in case the autosave wasn´t proc in the last row.

In [17]:
review_cleaned_rows(df)
files.download("cleaned_result.csv")


--- Row 0 ---

Original Text:

Identification. Small, head-body length 107 mm. Hair smooth and long (9 mm). Back orange, with gray
base. Belly white, contrasting with back (Fig. 11A). Tail not so long (125 mm) in comparison to
head-body length (116%), and with a small 6 mm brush at its tip. Skull with short rostrum, narrow
interorbital with finely bead ed supraorbital ledges (Fig. 12). Temporal ridges weak ly defined.
Zygomatic plates narrow and dorsal notch es shallow. Incisive foramina relatively short and wide,
their posterior margins do not reach the anterior margin of the first molars. Alisphenoid strut
present, and wide. Ectotympanic bulla small, exposing much of the medial periotic. Dentary with
small incisor tubercle.

Cleaned Text:

Identification. Small, head-body length 107 mm. Hair smooth and long (9 mm). Back orange with a gray
base. Belly white, contrasting with the back. The tail is not very long (125 mm) compared to the
head-body length (116%) and has a small 6 mm brus

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>